---
short_title: Erweiterte CSV-Verarbeitung
numbering:
    heading_1: true
    heading_2: true
    title: true
---

# Erweiterte CSV-Verarbeitung

In [Projekt CSV I](../../040-Projekt_CSV_I/000-Einleitung.md) haben Sie gelernt,
wie Sie Dateien öffnen und {term}`CSV`-Dateien mit `csv.reader` und
`csv.DictReader` einlesen. Die Grundlagen zu Zeichenkodierung, Trennzeichen und
Anführungszeichen stehen dort in

- [Dateien öffnen](../../040-Projekt_CSV_I/020-Dateien_öffnen.ipynb) und
- [CSV einlesen mit dem `csv`-Modul](../../040-Projekt_CSV_I/040-CSV_einlesen_Modul.ipynb).

Dieses Kapitel **wiederholt diese Grundlagen nicht**, sondern vertieft die
Fälle, die in der Praxis darüber hinausgehen:

- **alte Zeichenkodierungen** (`latin-1`) und den Umgang mit kaputten Zeichen,
- die **Feinsteuerung** von Anführungszeichen und Escaping,
- die **Umwandlung** von Textfeldern in sinnvolle Datentypen.

:::{seealso} Vorwissen
:icon: false

Die Grundlagen zum `csv`-Modul aus
[Projekt CSV I – CSV einlesen mit dem `csv`-Modul](../../040-Projekt_CSV_I/040-CSV_einlesen_Modul.ipynb)
werden hier vorausgesetzt.
:::


## Zeichenkodierung (Encoding)

Computer speichern Text als Bytes. Die **Zeichenkodierung** legt fest, wie
Bytes und Zeichen einander zugeordnet werden. Die Grundlagen – UTF-8 als
Standard und der `UnicodeDecodeError` beim falschen Encoding – kennen Sie aus
[Projekt CSV I](../../040-Projekt_CSV_I/020-Dateien_öffnen.ipynb).

Hier vertiefen wir den Fall, der in Bibliotheksdaten häufig vorkommt: **alte
Exporte in Latin-1** (ISO-8859-1). Latin-1 kennt genau 256 Zeichen und belegt
pro Zeichen ein Byte. Wird eine Latin-1-Datei fälschlich als UTF-8 gelesen,
entstehen entweder „kaputte“ Zeichen (Mojibake) oder ein `UnicodeDecodeError`.
Mit `errors="replace"` lassen sich solche Dateien trotzdem einlesen.


In [1]:
# Ein Text mit den Umlaut-nahen Zeichen ë und é
text = "Brontë, Charlotte - Café, Preis"
print(text)

# Latin-1 kann diese Zeichen darstellen (je 1 Byte)
als_bytes = text.encode("latin-1")
print("Latin-1-Bytes:", als_bytes)

# Wird fälschlich UTF-8 angenommen, scheitert das Dekodieren
try:
    als_bytes.decode("utf-8")
except UnicodeDecodeError as fehler:
    print("UnicodeDecodeError:", fehler)

# Mit errors="replace" bekommen wir wenigstens lesbare Ersatzzeichen
print(als_bytes.decode("utf-8", errors="replace"))

Brontë, Charlotte - Café, Preis
Latin-1-Bytes: b'Bront\xeb, Charlotte - Caf\xe9, Preis'
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xeb in position 5: invalid continuation byte
Bront�, Charlotte - Caf�, Preis


In [2]:
# UTF-8 kann dagegen alles – auch Emojis
daten = "Autor,Emoji\nMurakami, Haruki,📚\n"
roh = daten.encode("utf-8")
print("UTF-8-Bytes:", roh)
print("UTF-8 dekodiert:", roh.decode("utf-8"))
print("als Latin-1 gelesen:", roh.decode("latin-1", errors="replace"))

UTF-8-Bytes: b'Autor,Emoji\nMurakami, Haruki,\xf0\x9f\x93\x9a\n'
UTF-8 dekodiert: Autor,Emoji
Murakami, Haruki,📚

als Latin-1 gelesen: Autor,Emoji
Murakami, Haruki,ð



Beim Öffnen einer Datei legen wir die Kodierung deshalb immer **explizit**
fest:

```python
with open("daten.csv", encoding="utf-8", newline="") as f:
    leser = csv.reader(f)
```

- `encoding="utf-8"` ist die richtige Wahl für moderne Dateien.
- `encoding="latin-1"` hilft bei alten Exporten (z. B. aus
  Bibliothekssystemen), die kein UTF-8 schreiben.
- `errors="replace"` verhindert, dass ein einzelnes kaputtes Zeichen das
  gesamte Einlesen abbricht.
- `newline=""` überlässt die Zeilenende-Behandlung dem `csv`-Modul.


## Trennzeichen (Delimiter)

Nicht jede CSV-Datei trennt mit Komma. Im deutschsprachigen Raum ist das
**Semikolon** verbreitet, weil das Komma als Dezimaltrennzeichen dient. TSV
(*tab separated values*) nutzt den Tabulator. Wie Sie das Trennzeichen mit
`delimiter=...` setzen, haben Sie in
[Projekt CSV I](../../040-Projekt_CSV_I/040-CSV_einlesen_Modul.ipynb) gesehen.
Zur Erinnerung: Wird der Delimiter nicht angegeben, landet die ganze Zeile in
einer einzigen Spalte.


In [3]:
import csv
import io

semikolon = """isbn;author;sales
9780241951446;Murakami, Haruki;61
9783518587249;Boltanski, Luc;40
"""

# Falscher Delimiter (Komma): alles landet in einer Spalte
ohne_angabe = list(csv.reader(io.StringIO(semikolon)))
print("ohne Angabe:", ohne_angabe[0], "->", len(ohne_angabe[0]), "Spalte")

# Richtiger Delimiter (Semikolon)
richtig = list(csv.reader(io.StringIO(semikolon), delimiter=";"))
for zeile in richtig:
    print(zeile)

ohne Angabe: ['isbn;author;sales'] -> 1 Spalte
['isbn', 'author', 'sales']
['9780241951446', 'Murakami, Haruki', '61']
['9783518587249', 'Boltanski, Luc', '40']


```{note}
`csv.Sniffer` kann das Trennzeichen erraten – das haben Sie in
[Projekt CSV I](../../040-Projekt_CSV_I/040-CSV_einlesen_Modul.ipynb)
kennengelernt. In der Praxis geben Sie den Delimiter besser explizit an und
prüfen das Ergebnis.
```


## Anführungszeichen und Escaping

Enthält ein Feld selbst das Trennzeichen (z. B. ein Komma), wird es in doppelte
Anführungszeichen gesetzt. Auch Zeilenumbrüche sind innerhalb solcher Felder
erlaubt. Das `csv`-Modul verarbeitet das automatisch – solange wir nicht selbst
mit `split()` parsen.

In [4]:
kompliziert = '''titel,autor,kommentar
"Der neue Geist des Kapitalismus","Boltanski, Luc","Enthält ein Komma, im Text"
"Werke I-III","Sloterdijk, Peter","Mehrzeilig:
Zeile 2"

'''
for zeile in csv.reader(io.StringIO(kompliziert)):
    print(zeile)

['titel', 'autor', 'kommentar']
['Der neue Geist des Kapitalismus', 'Boltanski, Luc', 'Enthält ein Komma, im Text']
['Werke I-III', 'Sloterdijk, Peter', 'Mehrzeilig:\nZeile 2']
[]


Zur Feinsteuerung gibt es beim `csv`-Modul u. a.:

- `quotechar='"'` – welches Zeichen Felder umschließt,
- `quoting=csv.QUOTE_MINIMAL | QUOTE_ALL | QUOTE_NONE` – wann zitiert wird,
- `escapechar="\\"` – Zeichen zum Escapen, falls `QUOTE_NONE` genutzt wird,
- `doublequote=True` – ein verdoppeltes `""` steht für ein einzelnes `"`.

## Unregelmäßige Zeilen

Fehlen in einer Zeile Spalten oder kommen zusätzliche hinzu, erkennt man das
daran, dass die Feldanzahl nicht zum Kopf passt. Der `csv.reader` wirft dabei
**keinen** Fehler – das müssen wir selbst prüfen. Wie das geht und wie Sie
fehlerhafte Zeilen sauber protokollieren, steht in
[Projekt CSV I – Datenvalidierung](../../040-Projekt_CSV_I/045-Datenvalidierung.ipynb);
dort finden Sie auch die Funktion `pruefe_zeile()`.


## Felder in passende Typen umwandeln

CSV liefert zunächst **nur Text**. Für die Analyse müssen Zahlen zu `int` bzw.
`float` werden. Wir schreiben dazu eine kleine Funktion, die eine Zeile in ein
typisiertes Dictionary überführt. Fehlerhafte Werte werden mit `try`/`except`
abgefangen (ausführlich in
[Fehlerbehandlung & Performance](030-Fehlerbehandlung_Performance.ipynb)).

In [5]:
def parse_zeile(zeile):
    """Wandelt eine CSV-Zeile in ein typisiertes Dictionary um."""
    isbn, autor, jahr, titel, sales_year, sales = zeile
    return {
        "isbn": isbn,
        "author": autor,
        "year": int(jahr),          # Erscheinungsjahr als Zahl
        "title": titel,
        "sales_year": int(sales_year),
        "sales": int(sales),
    }

beispiel = ["9780241951446", "Murakami, Haruki", "1984",
            "Norwegian Wood", "2020", "61"]
satz = parse_zeile(beispiel)
print(satz)
print("Typ von year:", type(satz["year"]).__name__)

{'isbn': '9780241951446', 'author': 'Murakami, Haruki', 'year': 1984, 'title': 'Norwegian Wood', 'sales_year': 2020, 'sales': 61}
Typ von year: int


In [6]:
def parse_zeile_sicher(zeile):
    """Wie parse_zeile, gibt bei Fehlern aber ein Fehler-Dictionary zurück."""
    try:
        return parse_zeile(zeile)
    except (ValueError, TypeError) as fehler:
        return {"fehler": str(fehler), "rohdaten": zeile}

print(parse_zeile_sicher(beispiel))
print(parse_zeile_sicher(["x", "y", "keine Zahl", "t", "2020", "5"]))

{'isbn': '9780241951446', 'author': 'Murakami, Haruki', 'year': 1984, 'title': 'Norwegian Wood', 'sales_year': 2020, 'sales': 61}
{'fehler': "invalid literal for int() with base 10: 'keine Zahl'", 'rohdaten': ['x', 'y', 'keine Zahl', 't', '2020', '5']}


Diese Technik wenden wir nun auf die echte Datei an: Wir lesen die erste Zeile
mit dem `DictReader`, konvertieren die Felder und geben das Ergebnis aus. Die
kleine Hilfsfunktion `finde_datensatz()` liegt im Projekt als `csv_helfer.py`
und sucht die zentrale Testdatei unabhängig vom Arbeitsverzeichnis.


In [7]:
import sys
sys.path.append(".")          # erlaubt den Import des Helfers csv_helfer.py

import csv
from csv_helfer import finde_datensatz

pfad = finde_datensatz()
with open(pfad, encoding="utf-8", newline="") as f:
    leser = csv.DictReader(f)
    erste = next(leser)
print("Rohdaten:      ", erste)
print("Typ von sales: ", type(erste["sales"]).__name__)

typisiert = {
    "isbn": erste["isbn"],
    "author": erste["author"],
    "year": int(erste["year"]),
    "title": erste["title"],
    "sales_year": int(erste["sales_year"]),
    "sales": int(erste["sales"]),
}
print("Typisiert:     ", typisiert)
print("Typ von sales: ", type(typisiert["sales"]).__name__)


Rohdaten:       {'isbn': '9780241951446', 'author': 'Murakami, Haruki', 'year': '1984', 'title': 'Norwegian Wood', 'sales_year': '2020', 'sales': '61'}
Typ von sales:  str
Typisiert:      {'isbn': '9780241951446', 'author': 'Murakami, Haruki', 'year': 1984, 'title': 'Norwegian Wood', 'sales_year': 2020, 'sales': 61}
Typ von sales:  int


## Übung: Robust einlesen 💪

:::{exercise} Fehlerhafte Zeilen überspringen
:label: csv-robust

Lesen Sie den folgenden Text mit `csv.reader` und `delimiter=";"` ein. Geben Sie
für jede Datenzeile ein Dictionary aus. Zeilen, deren `verkauf`-Feld keine Zahl
ist, sollen mit einer Meldung **übersprungen** werden.

```python
text = "titel;autor;verkauf\nA;Murakami, Haruki;10\nB;Kafka, Franz;x\n"
```
:::

:::{solution} csv-robust
:class: dropdown

```python
import csv
import io

text = "titel;autor;verkauf\nA;Murakami, Haruki;10\nB;Kafka, Franz;x\n"
kopf = None
for nummer, zeile in enumerate(csv.reader(io.StringIO(text), delimiter=";"), start=1):
    if kopf is None:
        kopf = zeile
        continue
    daten = dict(zip(kopf, zeile))
    try:
        daten["verkauf"] = int(daten["verkauf"])
        print(nummer, daten)
    except ValueError:
        print(nummer, "übersprungen (keine Zahl):", daten)
```
:::

## Zusammenfassung

- Die **Kodierung** muss beim Öffnen explizit angegeben werden; UTF-8 ist der
  Standard, Latin-1 hilft bei alten Exporten.
- Der **Delimiter** ist nicht immer ein Komma; Semikolon und Tabulator sind
  verbreitet.
- Das `csv`-Modul behandelt **Anführungszeichen und Escaping** korrekt.
- **Unregelmäßige Zeilen** fallen nicht auf – wir müssen die Feldanzahl selbst
  prüfen.
- Textfelder werden mit `int()`/`float()` in **passende Typen** umgewandelt;
  fehlerhafte Werte fangen wir mit `try`/`except` ab.